# SeaDronesSee ResNet50 FP32 baseline trên Kaggle

Notebook này chạy nhánh FP32 baseline sạch của repo hiện tại:

- backbone `ResNet50`
- `2 class` với `binary_collapse_foreground=true`
- resolution `1080x1920`
- train FP32 trước, chưa đụng QAT

Flow:

1. clone hoặc pull repo đúng branch
2. tìm dataset root trên Kaggle Input
3. tạo runtime config đúng path hiện tại
4. smoke test ngắn
5. chạy 1 epoch có validation + benchmark


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/NguyenDucThang-tb/EchteAI.git"
BRANCH = "exp/resnet50-hawq-compiler"
REPO = Path("/kaggle/working/EchteAI")

def run(cmd, cwd=None):
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if REPO.exists():
    run(["git", "-C", str(REPO), "fetch", "--all"])
    run(["git", "-C", str(REPO), "checkout", BRANCH])
    run(["git", "-C", str(REPO), "pull", "origin", BRANCH])
else:
    run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)])

print("Repository:", REPO)
run(["git", "-C", str(REPO), "log", "--oneline", "-1"])


In [ ]:
from pathlib import Path

def find_dataset_root():
    preferred = [
        "/kaggle/input/notebooks/nguyenducthangtb/quantization-fasterrcnn/seadronessee",
        "/kaggle/input/seadronessee/compressed",
        "/kaggle/input/sds-dataset/compressed",
    ]
    for raw in preferred:
        candidate = Path(raw)
        if (candidate / "images/train").is_dir() and (candidate / "annotations/instances_train.json").is_file():
            return candidate

    for ann in Path("/kaggle/input").rglob("instances_train.json"):
        candidate = ann.parent.parent
        if (candidate / "images/train").is_dir() and (candidate / "annotations/instances_train.json").is_file():
            return candidate

    raise FileNotFoundError("Không tìm thấy dataset root có cấu trúc images/train + annotations/instances_train.json")

DATA_ROOT = find_dataset_root()
print("DATA_ROOT:", DATA_ROOT)
print("train_images:", DATA_ROOT / "images/train")
print("train_annotations:", DATA_ROOT / "annotations/instances_train.json")


In [ ]:
from pathlib import Path
import yaml

BASE_CONFIG = REPO / "configs" / "seadronessee_resnet50_fp32_2class_1080x1920.yaml"
RUNTIME_CONFIG = Path("/kaggle/working/runtime_resnet50_fp32_2class_1080x1920.yaml")
OUTPUT_ROOT = Path("/kaggle/working/resnet50_fp32_2class_1080x1920")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

cfg = yaml.safe_load(BASE_CONFIG.read_text())
cfg["dataset"]["train_images"] = str(DATA_ROOT / "images/train")
cfg["dataset"]["train_annotations"] = str(DATA_ROOT / "annotations/instances_train.json")
cfg["dataset"]["val_images"] = str(DATA_ROOT / "images/val")
cfg["dataset"]["val_annotations"] = str(DATA_ROOT / "annotations/instances_val.json")
cfg["dataset"]["test_images"] = str(DATA_ROOT / "images/val")
cfg["dataset"]["test_annotations"] = str(DATA_ROOT / "annotations/instances_val.json")
cfg["dataset"]["workers"] = 2
cfg["training"]["print_frequency"] = 50
cfg["output"]["directory"] = str(OUTPUT_ROOT)
cfg["output"]["fp32_best"] = str(OUTPUT_ROOT / "fp32_best.pt")
cfg["output"]["fp32_last"] = str(OUTPUT_ROOT / "fp32_last.pt")
cfg["output"]["qat_best"] = str(OUTPUT_ROOT / "qat_best.pt")
cfg["output"]["qat_last"] = str(OUTPUT_ROOT / "qat_last.pt")
cfg["output"]["int8_model"] = str(OUTPUT_ROOT / "selective_int8.pt")
cfg["output"]["evaluation_json"] = str(OUTPUT_ROOT / "evaluation.json")
cfg["output"]["benchmark_json"] = str(OUTPUT_ROOT / "benchmark.json")
cfg["output"]["epoch_benchmarks"] = str(OUTPUT_ROOT / "epoch_benchmarks.json")

RUNTIME_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print("Saved runtime config:", RUNTIME_CONFIG)
print(RUNTIME_CONFIG.read_text()[:2000])


## Smoke test rất ngắn

Cell này chỉ kiểm tra:

- dataloader chạy
- model train được
- không vỡ path hay môi trường

Nó bỏ validation + benchmark để phản hồi nhanh.

In [ ]:
cmd = [
    "python", "-u",
    "scripts/train_resnet50_fp32_baseline.py",
    "--config", str(RUNTIME_CONFIG),
    "--limit", "50",
    "--max-steps", "50",
    "--epochs-this-run", "1",
    "--skip-validation",
    "--skip-benchmark",
]
run(cmd, cwd=str(REPO))


## Chạy 1 epoch có validation + benchmark

Cell này là baseline ngắn nhưng đủ để xem train / val / benchmark có chạy thông suốt không.

Nếu muốn chạy lâu hơn thì tăng `--epochs-this-run`, bỏ `--limit` hoặc tăng `--max-steps`.

In [ ]:
cmd = [
    "python", "-u",
    "scripts/train_resnet50_fp32_baseline.py",
    "--config", str(RUNTIME_CONFIG),
    "--limit", "200",
    "--max-steps", "200",
    "--epochs-this-run", "1",
]
run(cmd, cwd=str(REPO))


In [ ]:
from pathlib import Path
import json

print("Output root:", OUTPUT_ROOT)
for p in sorted(OUTPUT_ROOT.glob("*")):
    print(p.name, p.stat().st_size / (1024 * 1024), "MB")

for name in ["epoch_benchmarks.json", "benchmark.json", "evaluation.json"]:
    p = OUTPUT_ROOT / name
    if p.exists():
        print(f"\n=== {name} ===")
        print(p.read_text()[:4000])


## Copy checkpoint ra chỗ dễ tải

Kaggle thường cho tải file trong `/kaggle/working` dễ hơn. Cell này copy checkpoint mới nhất ra thư mục export.

In [ ]:
from pathlib import Path
import shutil

EXPORT_DIR = Path("/kaggle/working/export_fp32_baseline")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for name in ["fp32_last.pt", "fp32_best.pt", "epoch_benchmarks.json"]:
    src = OUTPUT_ROOT / name
    if src.exists():
        shutil.copy2(src, EXPORT_DIR / src.name)
        print("Copied:", src, "->", EXPORT_DIR / src.name)
    else:
        print("Missing:", src)

print("EXPORT_DIR:", EXPORT_DIR)
